# Step 4 prep: acceptance checks for `case2s`

The checks every swap row must pass, run on the written `ct_pairs_{train,val,test}_case2s.parquet`. Function: `helpers/case2s_checks.py` (`flag_issues`, `run_acceptance`); tests on hand-made rows below it; then the run on the real files. Run after `03_case2_street_swap.ipynb`.

Every `case2s` row must pass these. Run on the files `03_case2_street_swap.ipynb` wrote (§13); the 1:9 check (#6) compares the full new set against the original `pairs`.

| Check | Type |
|---|---|
| `zip_a == zip_b`; same split; `id_a != id_b` | flagged |
| number in the new B == `number_a` | flagged |
| X's `street_a` vs the new B's street: no libpostal overlap | flagged |
| `(address_a, address_b)` unique among the swap rows | flagged |
| B matches `^\d+ \S`, no double spaces | flagged |
| per split, positives and negatives after §13 == counts in `pairs` (replacement, not added on top) | flagged |

**Known limits (2026-10-27, David's call):**
- #1 number and #3 street read the `number_*` / `street_*` columns made in §1 (the breakdown of the original
  address), not a re-parse of the built `address_b`. A §11 bug that builds the string wrong but keeps the columns
  right would pass.
- #3 libpostal overlap misses the same street when B carries extra text: `FARMINGTON AVE` → `{farmington avenue}`
  vs `Farmington Ave, Ste 2` → `{farmington avenue suite 2, …}`, no overlap. Fixing it needs the `road` part of B
  (e.g. `parse_address`); not done (time). Same limit applies to §3.

## Setup

In [ ]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.pairs import load_all_pairs
from helpers.case2s_checks import flag_issues, load_written, run_acceptance

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

## Load
**Does:** `pairs` = the original pair set (before the swaps); `written` = the three new files in one frame.  
**Output:** `pairs`, `written`

In [ ]:
# Load: original pairs and the written case2s files
pairs = load_all_pairs("overture")
written = load_written()
print(pairs.shape, written.shape)

## Tests on hand-made rows
**Does:** a 1:9 frame (per split: 1 positive, 8 easy negatives, 1 swap); each bad edit must be flagged by its own check.

In [ ]:
# Acceptance tests: a 1:9 frame (per split: 1 positive, 8 easy negatives, 1 swap); each bad edit must be flagged by its own check
splits = ["train", "val", "test"]
numbers = {"train": "12", "val": "14", "test": "16"}

def make_row(split, id_a, id_b, label, neg_type):
    n = numbers[split]
    return {
        "id_a": id_a, "id_b": id_b,
        "address_a": f"{n} Main Street", "address_b": f"{n} Maple Avenue",
        "street_a": "Main Street", "street_b": "Maple Avenue",
        "zip_a": "06901", "zip_b": "06901",
        "number_a": n, "number_b": n,
        "split": split, "label": label, "neg_type": neg_type,
    }

def make_frame():
    rows = []
    for s in splits:
        rows.append(make_row(s, f"p_{s}", f"fp_{s}", 1, "case0"))
        rows += [make_row(s, f"e{k}_{s}", f"fe{k}_{s}", 0, "case3") for k in range(8)]
        rows.append(make_row(s, f"x_{s}", f"y_{s}", 0, "case2s"))
    return pd.DataFrame(rows)

# `pairs` stand-in = the set before §13: the same frame, with each swap row still an easy negative (same counts, same ids)
pairs_t = make_frame().replace({"neg_type": {"case2s": "case3"}})

def edit(**cols):
    def apply(frame, i):
        for col, value in cols.items():
            frame.at[i, col] = value
    return apply

def add_swap_on_top(frame, i):
    # the "add on top" bug: an extra swap row with no easy negative removed (ids known, addresses unique -> only #6 should fire)
    row = frame.loc[i].copy()
    row["address_a"], row["address_b"] = "12 Oak Street", "12 Elm Avenue"
    frame.loc[len(frame)] = row

cases = {
    "good frame":          (None,                    edit()),
    "id_a == id_b":        ("#1 id_a equals id_b",   edit(id_b="x_train")),
    "zip mismatch":        ("#1 zip_a doesn't",      edit(zip_b="06902")),
    "number mismatch":     ("#1 number_a doesn't",   edit(number_b="99")),
    "Y in another split":  ("#2",                    edit(id_b="y_val")),
    "libpostal overlap":   ("#3",                    edit(street_b="Main St")),
    "duplicate pair":      ("#4",                    edit(address_a="14 Main Street", address_b="14 Maple Avenue")),
    "B has no number":     ("#5",                    edit(address_b="Maple Avenue")),
    "no number on A or B": (None,                    edit(address_a="Main Street", address_b="Maple Avenue", number_a=None, number_b=None)),
    "split not 1:9":       ("#6",                    edit(label=1)),
    "swap added on top":   ("#6",                    add_swap_on_top),
}

for name, (expected, mutate) in cases.items():
    frame = make_frame()
    swap_row = frame.index[(frame["split"] == "train") & (frame["neg_type"] == "case2s")][0]
    mutate(frame, swap_row)
    messages = [issue[0] for issue in flag_issues(frame, pairs_t)]
    ok = not messages if expected is None else any(expected in m for m in messages)
    print("PASS" if ok else "FAIL", name, "|", [m.split(". Rows")[0] for m in messages])


## Run on the written files
**Does:** `case2s` rows only: real `case2` rows are not swaps and legitimately fail #1 number / ZIP, #3, #5 (they are dropped from both frames, so #6 still balances).  
**Done when:** 0 issues.

In [ ]:
# Acceptance run on the written files
issues = run_acceptance(written, pairs)
for message, sample in issues:
    print(message)
    display(sample)
print(len(issues), "issues")